In [0]:
# ============================================================
# Job Parameters
# ============================================================
from datetime import datetime

# Create job parameters (widgets)
dbutils.widgets.text("run_date", "2026-07-10", "Run Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"], "Environment")
dbutils.widgets.text("s3_bucket", "retail-data-lake-pj2026det", "S3 Bucket Name")
dbutils.widgets.text("catalog_name", "retail-data-lake-catalog", "Unity Catalog Name")

# Get parameter values
run_date = dbutils.widgets.get("run_date")
environment = dbutils.widgets.get("environment")
s3_bucket = dbutils.widgets.get("s3_bucket")
catalog_name = dbutils.widgets.get("catalog_name")

# Display parameters
print("=" * 60)
print("JOB PARAMETERS")
print("=" * 60)
print(f"Run Date:      {run_date}")
print(f"Environment:   {environment}")
print(f"S3 Bucket:     s3://{s3_bucket}")
print(f"Catalog:       {catalog_name}")
print(f"Execution Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# Create bronze schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.bronze")
print(f"\n✓ Schema {catalog_name}.bronze ready")


In [0]:
# ============================================================
# S3 Connection Check
# ============================================================
import sys

print("Verifying S3 connection...")
print("=" * 60)

try:
    # Test S3 bucket access
    s3_raw_path = f"s3://{s3_bucket}/raw/"
    files = dbutils.fs.ls(s3_raw_path)
    
    print(f"✓ S3 connection successful")
    print(f"  Bucket: s3://{s3_bucket}")
    print(f"  Path: {s3_raw_path}")
    print(f"  Directories found: {len(files)}")
    print("\nAvailable source directories:")
    for f in files:
        print(f"  - {f.name}")
    
    print("=" * 60)
    print("✓ All prerequisites validated. Ready to ingest data.")
    
except Exception as e:
    print(f"✗ S3 connection failed: {str(e)}")
    print("\nPossible causes:")
    print("  1. S3 bucket name is incorrect")
    print("  2. IAM permissions are missing")
    print("  3. Bucket does not exist or is not accessible")
    print("  4. Network connectivity issue")
    print("=" * 60)
    print("✗ FAILED - Cannot proceed with ingestion")
    sys.exit(1)  # Exit with error code

In [0]:
%sql
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/pos/transactions/pos_transactions_20260710.csv',
  format => 'csv',
  header => true
)


In [0]:
# Save online_orders to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.pos_transactions_20260710")

In [0]:
%sql
-- Read e-commerce online orders JSON
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/ecommerce/online_orders_20260710.json',
  format => 'json'
)

In [0]:
# Save online_orders to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.online_orders_20260710")

In [0]:
%sql
-- Read web clickstream log files (stored as text)
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/web/clickstream/web_clickstream_20260710.log',
  format => 'text'
)

In [0]:
# Save web_clickstream to bronze (as raw text for parsing in Silver)
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.web_clickstream_20260710")

In [0]:
%sql
-- Read CRM customer master JSON
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/crm/customer_master.json',
  format => 'json'
)

In [0]:
# Save customer_master to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.customer_master")

In [0]:
%sql
-- Read CRM loyalty members CSV
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/crm/loyalty_members.csv',
  format => 'csv',
  header => true
)

In [0]:
# Save loyalty_members to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.loyalty_members")

In [0]:
%sql
-- Read CRM coupon dispatches JSON
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/crm/coupon_dispatches.json',
  format => 'json'
)

In [0]:
# Save coupon_dispatches to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.coupon_dispatches")

In [0]:
%sql
-- Read ERP product master CSV
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/erp/product_master.csv',
  format => 'csv',
  header => true
)

In [0]:
# Save product_master to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.product_master")

In [0]:
%sql
-- Read ERP category master CSV
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/erp/category_master.csv',
  format => 'csv',
  header => true
)

In [0]:
# Save category_master to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.category_master")

In [0]:
%sql
-- Read ERP supplier master CSV
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/erp/supplier_master.csv',
  format => 'csv',
  header => true
)

In [0]:
# Save supplier_master to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.supplier_master")

In [0]:
%sql
-- Read ERP store master CSV
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/erp/store_master.csv',
  format => 'csv',
  header => true
)

In [0]:
# Save store_master to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.store_master")

In [0]:
%sql

-- Read ERP inventory balance (try CSV format due to parquet read error)
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/erp/inventory_balance.parquet',
  format => 'parquet',
  header => true
)

In [0]:
# Save inventory_balance to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.inventory_balance")

In [0]:
%sql
-- Read reference retail calendar CSV
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/reference/retail_calendar.csv',
  format => 'csv',
  header => true
)

In [0]:
# Save retail_calendar to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.retail_calendar")

In [0]:
%sql
-- Read reference holiday calendar CSV
SELECT * FROM read_files(
  's3://retail-data-lake-pj2026det/raw/reference/holiday_calendar.csv',
  format => 'csv',
  header => true
)

In [0]:
# Save holiday_calendar to bronze
_sqldf.write.format("delta").mode("overwrite").saveAsTable("`retail-data-lake-catalog`.bronze.holiday_calendar")

In [0]:
# Check which bronze tables exist
tables_to_check = [
    'pos_transactions_20260710',
    'online_orders_20260710',
    'web_clickstream_20260710',
    'customer_master',
    'loyalty_members',
    'coupon_dispatches',
    'product_master',
    'category_master',
    'supplier_master',
    'store_master',
    'inventory_balance',
    'retail_calendar',
    'holiday_calendar'
]

print("Bronze Table Existence Check:")
print("=" * 60)

# List to store tables that exist in bronze
existing_tables = []
missing_tables = []

for table in tables_to_check:
    full_name = f"`retail-data-lake-catalog`.bronze.{table}"
    exists = spark.catalog.tableExists(full_name)
    status = "✓ EXISTS" if exists else "✗ MISSING"
    print(f"{status:12} | {table}")
    
    if exists:
        existing_tables.append(table)
    else:
        missing_tables.append(table)

print("=" * 60)  # Separator line for readability
print(f"\nSummary: {len(existing_tables)}/{len(tables_to_check)} tables exist")

if missing_tables:
    print(f"\n⚠ Missing tables: {', '.join(missing_tables)}")
    print("   Run the corresponding ingestion cells to create them.")
else:
    print("\n✓ All tables successfully created!")

In [0]:
%sql
-- Validate all Bronze tables were created successfully
SELECT 
    'pos_transactions_20260710' as table_name,
    COUNT(*) as row_count,
    'Transaction data' as description
FROM `retail-data-lake-catalog`.bronze.pos_transactions_20260710

UNION ALL

SELECT 
    'online_orders_20260710',
    COUNT(*),
    'E-commerce orders'
FROM `retail-data-lake-catalog`.bronze.online_orders_20260710

UNION ALL

SELECT 
    'web_clickstream_20260710',
    COUNT(*),
    'Web clickstream logs'
FROM `retail-data-lake-catalog`.bronze.web_clickstream_20260710

UNION ALL

SELECT 
    'customer_master',
    COUNT(*),
    'Customer master data'
FROM `retail-data-lake-catalog`.bronze.customer_master

UNION ALL

SELECT 
    'loyalty_members',
    COUNT(*),
    'Loyalty program members'
FROM `retail-data-lake-catalog`.bronze.loyalty_members

UNION ALL

SELECT 
    'coupon_dispatches',
    COUNT(*),
    'Coupon dispatch records'
FROM `retail-data-lake-catalog`.bronze.coupon_dispatches

UNION ALL

SELECT 
    'product_master',
    COUNT(*),
    'Product master data'
FROM `retail-data-lake-catalog`.bronze.product_master

UNION ALL

SELECT 
    'category_master',
    COUNT(*),
    'Category master data'
FROM `retail-data-lake-catalog`.bronze.category_master

UNION ALL

SELECT 
    'supplier_master',
    COUNT(*),
    'Supplier master data'
FROM `retail-data-lake-catalog`.bronze.supplier_master

UNION ALL

SELECT 
    'store_master',
    COUNT(*),
    'Store master data'
FROM `retail-data-lake-catalog`.bronze.store_master

UNION ALL

SELECT 
    'inventory_balance',
    COUNT(*),
    'Inventory balance'
FROM `retail-data-lake-catalog`.bronze.inventory_balance

UNION ALL

SELECT 
    'retail_calendar',
    COUNT(*),
    'Retail calendar'
FROM `retail-data-lake-catalog`.bronze.retail_calendar

UNION ALL

SELECT 
    'holiday_calendar',
    COUNT(*),
    'Holiday calendar'
FROM `retail-data-lake-catalog`.bronze.holiday_calendar

ORDER BY table_name